01. Limpiar y ordenar TiemposRuta de la carpeta 

In [1]:
#Librerias y funciones
import os
import pandas as pd

# Ruta de la carpeta de entrada
ruta_inputs = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Inputs"

# Buscar el primer archivo Excel o CSV en la carpeta
archivos = [f for f in os.listdir(ruta_inputs) if f.endswith(('.xlsx', '.xls', '.csv'))]

if archivos:
    archivo = archivos[0]
    ruta_archivo = os.path.join(ruta_inputs, archivo)
    if archivo.endswith('.csv'):
        tiempos_ruta_abasto = pd.read_csv(ruta_archivo, header=6)
    else:
        tiempos_ruta_abasto = pd.read_excel(ruta_archivo, header=6)
    # Elimina todas las columnas que contienen 'Unnamed:'
    tiempos_ruta_abasto = tiempos_ruta_abasto.loc[:, ~tiempos_ruta_abasto.columns.str.contains('^Unnamed')]

    # Separar todas las columnas en _A y _B según el criterio solicitado
    df_sep = pd.DataFrame()
    for col in tiempos_ruta_abasto.columns:
        col_A = []
        col_B = []
        for i, val in enumerate(tiempos_ruta_abasto[col]):
            if i % 2 == 0:
                col_A.append(val)
            else:
                col_B.append(val)
        # Rellenar para igualar longitudes
        max_len = max(len(col_A), len(col_B))
        col_A += [None] * (max_len - len(col_A))
        col_B += [None] * (max_len - len(col_B))
        df_sep[col + '_A'] = col_A
        df_sep[col + '_B'] = col_B

    # Renombrar columnas según lo solicitado
    renombrar = {
        'Tipo Unidad/Partidas_A': 'Tipo de Unidad',
        'Ruta/Nombre Ruta_A': 'Ruta',
        'Ruta/Nombre Ruta_B': 'Nombre Ruta',
        'Hoja Ruta/ Placas_A': 'SAP',
        'Hoja Ruta/ Placas_B': 'Placa',
        'Cierre': 'Cierre',
        'Total Escaneos_A': 'Escaneos'
        
    }
    # Seleccionar solo las columnas que se van a conservar y renombrar
    columnas_finales = [col for col in renombrar if col in df_sep.columns]
    if 'Cierre' in tiempos_ruta_abasto.columns:
        df_sep['Cierre'] = tiempos_ruta_abasto['Cierre'].iloc[::2].reset_index(drop=True)
        # Insertar 'Cierre' antes de 'Total Escaneos_A' si existe
        if 'Total Escaneos_A' in columnas_finales:
            idx = columnas_finales.index('Total Escaneos_A')
            columnas_finales.insert(idx, 'Cierre')
        else:
            columnas_finales.append('Cierre')
    df_sep = df_sep[columnas_finales].rename(columns=renombrar)

    # Crear columna 'Ruta Conca' concatenando año y ruta (año fijo 2026)
    df_sep.insert(
        df_sep.columns.get_loc('Nombre Ruta'),  # Inserta antes de 'Nombre Ruta'
        'Ruta Conca',
        '2026' + df_sep['Ruta'].astype(str)
    )

    # Quitar los primeros 3 ceros de la columna SAP
    if 'SAP' in df_sep.columns:
        df_sep['SAP'] = df_sep['SAP'].astype(str).str.replace('^000', '', regex=True)

    print(f"Archivo leído: {archivo}")
    print(f"Total de columnas: {len(df_sep.columns)}")
    print("Nombres de columnas:", list(df_sep.columns))
else:
    print("No se encontró ningún archivo Excel o CSV en la carpeta de Inputs.")

Archivo leído: Copia de TiemposRutaExcel 04-10may.xls
Total de columnas: 8
Nombres de columnas: ['Tipo de Unidad', 'Ruta', 'Ruta Conca', 'Nombre Ruta', 'SAP', 'Placa', 'Cierre', 'Escaneos']


02. Quitar y renombrar columnas, quitar todo lo que no sea SUCURSAL o COMPLEMENTO

In [2]:
# Filtrar filas donde 'Nombre Ruta' contenga 'SUCUR' o 'COMP', ignorando mayúsculas/minúsculas
filtro = df_sep['Nombre Ruta'].astype(str).str.contains('SUCUR|COMP', case=False, na=False)

# Filas que cumplen el filtro
df_filtrado = df_sep[filtro].copy()
# Filas que NO cumplen el filtro (las que se van a borrar)
df_borradas = df_sep[~filtro].copy()

# Limpiar filas vacías, ceros o nan en 'Nombre Ruta'
df_filtrado = df_filtrado[
    (df_filtrado['Nombre Ruta'].notna()) &
    (df_filtrado['Nombre Ruta'].astype(str).str.strip() != '') &
    (df_filtrado['Nombre Ruta'].astype(str) != '0')
].reset_index(drop=True)

df_borradas = df_borradas[
    (df_borradas['Nombre Ruta'].notna()) &
    (df_borradas['Nombre Ruta'].astype(str).str.strip() != '') &
    (df_borradas['Nombre Ruta'].astype(str) != '0')
].reset_index(drop=True)

print(f"Filas originales: {len(df_sep)}")
print(f"Filas después del filtrado: {len(df_filtrado)}")
print(f"Filas eliminadas: {len(df_borradas)}")

print("Ejemplo de filas eliminadas:")

Filas originales: 532
Filas después del filtrado: 531
Filas eliminadas: 1
Ejemplo de filas eliminadas:


03. Quitar SUCURSAL FORANEO

In [3]:
#Eliminar filas con 'SUCURSAL FORANEO' en 'Nombre Ruta'
def eliminar_sucursal_foraneo(df, columna='Nombre Ruta'):
    # Filtrar filas a eliminar
    filtro_foraneo = df[columna].astype(str).str.contains('SUCURSAL FORANEO', case=False, na=False)
    df_borrados = df[filtro_foraneo].copy()
    df_filtrado = df[~filtro_foraneo].copy()
    
    print(f"Filas originales: {len(df)}")
    print(f"Filas después de eliminar 'SUCURSAL FORANEO': {len(df_filtrado)}")
    print(f"Filas eliminadas: {len(df_borrados)}")
    print("Ejemplo de filas eliminadas:")
    
    return df_filtrado, df_borrados

# Uso del módulo con df_filtrado
df_filtrado_sf, df_borrados_sf = eliminar_sucursal_foraneo(df_filtrado)

Filas originales: 531
Filas después de eliminar 'SUCURSAL FORANEO': 288
Filas eliminadas: 243
Ejemplo de filas eliminadas:


4. Ordenas A>Z Placa y Cierre

In [4]:
# Ordenar df_filtrado por las columnas 'Placa' y 'Cierre' de la A a la Z
df_filtrado_sf = df_filtrado_sf.sort_values(by=['Placa', 'Cierre'], ascending=[True, True]).reset_index(drop=True)

5. Encontrar datos vacíos en columna SAP 

In [5]:
# Crear un nuevo DataFrame con filas donde la columna 'SAP' esté vacía, sea nan, 'nan', 'Nan' o '0'
sin_SAP = df_filtrado_sf[
    df_filtrado_sf['SAP'].isna() |
    (df_filtrado_sf['SAP'].astype(str).str.lower().isin(['nan'])) |
    (df_filtrado_sf['SAP'].astype(str).str.strip() == '') |
    (df_filtrado_sf['SAP'].astype(str).str.strip() == '0')
].copy()

print(f"Total de filas sin SAP: {len(sin_SAP)}")

Total de filas sin SAP: 1


6. Busqueda en SART (Sin SAP)

In [6]:
# Busqueda en SART (robusta ante cambios de IDs en la pagina)
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, NoSuchDriverException, WebDriverException

# --- Configuracion ---
USUARIO = "IGCAMPOSG"
CONTRASENA = "Carcachita22$"
URLS_LOGIN = [
    "https://sartt.truper.com/sartt/login.jspx",
    "http://sartt.truper.com/sartt/login.jspx"
]
URLS_REPORTE = [
    "https://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx",
    "http://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx"
]

def _es_sesion_cerrada(exc):
    """Devuelve True si el error es por sesion invalida o navegador cerrado."""
    msg = str(exc).lower()
    return any(kw in msg for kw in ["invalid session id", "browser has closed", "not connected to devtools", "disconnected"])

def esperar_elemento(wait, locators, clickable=False, timeout_msg="No se encontro el elemento esperado"):
    """Prueba varios selectores para el mismo elemento y devuelve el primero que funcione."""
    ultimo_error = None
    for by, selector in locators:
        try:
            condicion = EC.element_to_be_clickable((by, selector)) if clickable else EC.presence_of_element_located((by, selector))
            return wait.until(condicion)
        except Exception as exc:
            ultimo_error = exc
    raise TimeoutException(f"{timeout_msg}. Ultimo error: {ultimo_error}")

def abrir_url_robusta(driver, wait, urls, etiqueta):
    """Abre una URL probando variantes (https/http) y valida que no quede en data:."""
    ultimo_error = None
    for intento, url in enumerate(urls, start=1):
        try:
            driver.get(url)
            wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
            time.sleep(1)
            print(f"{etiqueta} intento {intento}: {url}")
            print(f"URL actual: {driver.current_url} | Titulo: {driver.title}")
            if not driver.current_url.startswith("data:"):
                return
        except Exception as exc:
            ultimo_error = exc
    raise TimeoutException(f"No fue posible abrir {etiqueta} sin quedar en data:. Ultimo error: {ultimo_error}")

def crear_driver_web():
    """Intenta Edge con rutas locales del driver y usa Chrome como respaldo."""
    edge_options = webdriver.EdgeOptions()
    edge_options.add_argument("--start-maximized")
    edge_options.add_argument("--ignore-certificate-errors")
    edge_options.add_argument("--inprivate")

    rutas_edge_driver = [
        os.environ.get('MSEDGEDRIVER_PATH'),
        r"C:\WebDriver\msedgedriver.exe",
        r"C:\tools\msedgedriver.exe",
        r"C:\Program Files\msedgedriver.exe",
        r"C:\Program Files (x86)\msedgedriver.exe"
    ]

    for ruta in rutas_edge_driver:
        if ruta and os.path.exists(ruta):
            try:
                print(f"Usando EdgeDriver local: {ruta}")
                return webdriver.Edge(service=EdgeService(executable_path=ruta), options=edge_options), 'edge'
            except Exception as exc:
                print(f"No se pudo iniciar Edge con {ruta}: {exc}")

    try:
        return webdriver.Edge(options=edge_options), 'edge'
    except (NoSuchDriverException, WebDriverException) as exc:
        print(f"No se pudo iniciar Edge automatico: {exc}")
        print("Fallback a Chrome para no detener el proceso...")
        chrome_options = webdriver.ChromeOptions()
        chrome_options.add_argument("--start-maximized")
        chrome_options.add_argument("--ignore-certificate-errors")
        return webdriver.Chrome(service=ChromeService(), options=chrome_options), 'chrome'

# Obtener rutas unicas del dataframe sin_SAP
if 'sin_SAP' in locals() and not sin_SAP.empty:
    rutas_sin_sap = sin_SAP['Ruta'].dropna().astype(str).unique().tolist()
    rutas_texto = '\n'.join(rutas_sin_sap)
else:
    rutas_sin_sap = ["99897"]
    rutas_texto = "99897"

driver, navegador_activo = crear_driver_web()
print(f"Navegador Selenium activo: {navegador_activo}")
wait = WebDriverWait(driver, 35)

# DataFrame para almacenar resultados
resultados_completos = pd.DataFrame()

try:
    # 1) Login
    abrir_url_robusta(driver, wait, URLS_LOGIN, "login SART")

    usuario_input = esperar_elemento(
        wait,
        [
            (By.ID, "loginForm:j_username"),
            (By.NAME, "j_username"),
            (By.CSS_SELECTOR, "input[id*='j_username']"),
            (By.CSS_SELECTOR, "input[name*='j_username']"),
            (By.CSS_SELECTOR, "input[type='text']")
        ],
        clickable=True,
        timeout_msg="No se encontro el campo de usuario"
    )
    usuario_input.clear()
    usuario_input.send_keys(USUARIO)

    password_input = esperar_elemento(
        wait,
        [
            (By.ID, "loginForm:j_password"),
            (By.NAME, "j_password"),
            (By.CSS_SELECTOR, "input[id*='j_password']"),
            (By.CSS_SELECTOR, "input[name*='j_password']"),
            (By.CSS_SELECTOR, "input[type='password']")
        ],
        clickable=True,
        timeout_msg="No se encontro el campo de contrasena"
    )
    password_input.clear()
    password_input.send_keys(CONTRASENA)

    boton_login = esperar_elemento(
        wait,
        [
            (By.ID, "loginForm:j_id46"),
            (By.CSS_SELECTOR, "input[id^='loginForm:j_id']"),
            (By.CSS_SELECTOR, "input.iceCmdBtn[value='Entrar']"),
            (By.CSS_SELECTOR, "input[type='submit'][value='Entrar']"),
            (By.CSS_SELECTOR, "input[type='submit']")
        ],
        clickable=True,
        timeout_msg="No se encontro el boton de login"
    )
    try:
        boton_login.click()
    except Exception:
        driver.execute_script("arguments[0].click();", boton_login)

    # Validar entrada al sitio: dashboard o URL ya autenticada
    wait.until(lambda d: (not d.current_url.startswith("data:")) and (("dashboard" in d.page_source.lower()) or ("reporte" in d.current_url.lower()) or ("principal" in d.current_url.lower())))
    print("Login exitoso en SART.")

    # 2) Navegacion al reporte
    abrir_url_robusta(driver, wait, URLS_REPORTE, "reporte SART")

    # 3) Seleccionar CIAT y pegar rutas (IDs dinamicos con fallback)
    select_element = esperar_elemento(
        wait,
        [
            (By.ID, "forma:icePnlTbSet:0:j_id202"),
            (By.NAME, "forma:icePnlTbSet:0:j_id202"),
            (By.CSS_SELECTOR, "select.iceSelOneMnu[id$='j_id202']"),
            (By.ID, "forma:icePnlTbSet:0:j_id200"),
            (By.CSS_SELECTOR, "select[id$='j_id200']"),
            (By.CSS_SELECTOR, "select[id*='icePnlTbSet:0']")
        ],
        clickable=True,
        timeout_msg="No se encontro el selector de tipo de ruta"
    )

    try:
        Select(select_element).select_by_value("2")  # CIAT
    except NoSuchElementException:
        # Fallback por texto visible, por si cambiaron el value
        Select(select_element).select_by_visible_text("CIAT")

    time.sleep(1)

    cuadro_rutas = esperar_elemento(
        wait,
        [
            (By.ID, "forma:icePnlTbSet:0:j_id206"),
            (By.NAME, "forma:icePnlTbSet:0:j_id206"),
            (By.CSS_SELECTOR, "textarea.iceInpTxtArea[id$='j_id206']"),
            (By.ID, "forma:icePnlTbSet:0:j_id204"),
            (By.CSS_SELECTOR, "textarea[id$='j_id204']"),
            (By.CSS_SELECTOR, "textarea[id*='icePnlTbSet:0']")
        ],
        clickable=True,
        timeout_msg="No se encontro el cuadro para pegar rutas"
    )
    cuadro_rutas.clear()
    cuadro_rutas.send_keys(rutas_texto)
    cuadro_rutas.send_keys(Keys.TAB)

    print(f"Rutas enviadas a SART: {len(rutas_sin_sap)}")
    print("Da clic en 'Buscar' manualmente y espera a que cargue la tabla...")

    # 4) Esperar tabla de resultados
    id_tabla_resultados = "forma:icePnlTbSet:0:rutas"
    tabla_elemento = esperar_elemento(
        wait,
        [
            (By.ID, id_tabla_resultados),
            (By.CSS_SELECTOR, "table[id$='rutas']")
        ],
        clickable=False,
        timeout_msg="No se encontro la tabla de resultados"
    )

    tabla_html = tabla_elemento.get_attribute('outerHTML')
    from io import StringIO
    resultados_completos = pd.read_html(StringIO(tabla_html))[0]

    if isinstance(resultados_completos.columns, pd.MultiIndex):
        resultados_completos.columns = ['_'.join(col).strip() for col in resultados_completos.columns.values]
    resultados_completos.columns = resultados_completos.columns.str.replace(r'[^\w\s]', '', regex=True)
    resultados_completos.columns = resultados_completos.columns.str.strip()

    print(f"Rutas encontradas en sin_SAP: {rutas_sin_sap}")
    print(f"Total de rutas: {len(rutas_sin_sap)}")
    if not resultados_completos.empty:
        print("\n--- RESULTADOS ENCONTRADOS ---")
    else:
        print("\n--- NO SE ENCONTRARON RESULTADOS ---")

except Exception as e:
    if _es_sesion_cerrada(e):
        print("\u26a0\ufe0f Navegador cerrado inesperadamente. Continuando al siguiente paso con los datos actuales.")
        print(f"   resultados_completos: {len(resultados_completos)} filas conservadas.")
    elif isinstance(e, TimeoutException):
        print(f"Error de timeout en SART: {e}")
        try:
            print(f"URL actual: {driver.current_url}")
            print(f"Titulo de pagina: {driver.title}")
            html_debug = os.path.join(os.getcwd(), "sart_debug_login.html")
            with open(html_debug, "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"Se guardo HTML de diagnostico en: {html_debug}")
        except Exception:
            print("No se pudo obtener informacion adicional del navegador.")
    else:
        print(f"Error inesperado en SART: {e}")
        try:
            print(f"URL actual: {driver.current_url}")
            print(f"Titulo de pagina: {driver.title}")
        except Exception:
            print("No se pudo obtener informacion adicional del navegador.")

finally:
    if 'driver' in locals():
        try:
            driver.quit()
        except Exception:
            pass

No se pudo iniciar Edge automatico: Message: Unable to obtain driver for MicrosoftEdge; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location

Fallback a Chrome para no detener el proceso...
Navegador Selenium activo: chrome
login SART intento 2: http://sartt.truper.com/sartt/login.jspx
URL actual: http://sartt.truper.com/sartt/login.jspx | Titulo: SARTT:::::::::Sistema de Administración de Rutas y Transportes
Login exitoso en SART.
reporte SART intento 2: http://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx
URL actual: http://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx | Titulo: SARTT
Rutas enviadas a SART: 1
Da clic en 'Buscar' manualmente y espera a que cargue la tabla...
Rutas encontradas en sin_SAP: ['73244']
Total de rutas: 1

--- RESULTADOS ENCONTRADOS ---


7. Completar SAP faltantes con datos de SARTT

In [7]:
# Verificar que tenemos los DataFrames necesarios
if 'df_filtrado_sf' in locals() and 'resultados_completos' in locals() and not resultados_completos.empty:
    tiempos_limpio = df_filtrado_sf.copy()
    
    # Limpiar columna Ruta en tiempos_limpio
    tiempos_limpio['Ruta'] = tiempos_limpio['Ruta'].astype(str).str.lstrip('0')
    tiempos_limpio['Ruta'] = tiempos_limpio['Ruta'].replace('', '0')
    
    # Identificar SAP vacíos ANTES de completar
    sap_vacios_antes = tiempos_limpio[
        tiempos_limpio['SAP'].isna() |
        (tiempos_limpio['SAP'].astype(str).str.lower().isin(['nan'])) |
        (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
        (tiempos_limpio['SAP'].astype(str).str.strip() == '0')
    ].copy()
    
    print(f"Total de filas con SAP vacío antes: {len(sap_vacios_antes)}")
    
    # Buscar columnas en resultados_completos
    columna_ruta_sartt = None
    columna_sap_sartt = None
    
    print(f"\nColumnas disponibles en resultados_completos: {list(resultados_completos.columns)}")
    
    for col in resultados_completos.columns:
        col_upper = col.upper()
        if 'CIAT' in col_upper and 'RUTA' in col_upper:
            columna_ruta_sartt = col
            print(f"✅ Columna de Ruta SARTT encontrada: '{col}'")
        elif 'SAP' in col_upper and 'RUTA' in col_upper:
            columna_sap_sartt = col
            print(f"✅ Columna de SAP SARTT encontrada: '{col}'")
    
    if columna_ruta_sartt and columna_sap_sartt:
        # CLAVE: Limpiar también los ceros a la izquierda en resultados_completos
        resultados_completos_limpio = resultados_completos.copy()
        resultados_completos_limpio[columna_ruta_sartt] = (
            resultados_completos_limpio[columna_ruta_sartt]
            .astype(str)
            .str.lstrip('0')
            .replace('', '0')
        )
        
        # Crear mapeo con rutas limpias
        mapeo_sap = dict(zip(
            resultados_completos_limpio[columna_ruta_sartt].astype(str),
            resultados_completos_limpio[columna_sap_sartt].astype(str)
        ))
        
        print(f"\nMapeo creado con {len(mapeo_sap)} rutas")
        print(f"Ejemplo de mapeo: {dict(list(mapeo_sap.items())[:3])}")
        
        # Crear máscara de SAP vacíos
        mascara_sap_vacio = (
            tiempos_limpio['SAP'].isna() |
            (tiempos_limpio['SAP'].astype(str).str.lower() == 'nan') |
            (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
            (tiempos_limpio['SAP'].astype(str).str.strip() == '0')
        )
        
        # Completar SAP usando el mapeo (método vectorizado, más rápido)
        rutas_a_completar = tiempos_limpio.loc[mascara_sap_vacio, 'Ruta'].astype(str)
        nuevos_sap = rutas_a_completar.map(mapeo_sap)
        
        # Filtrar solo los SAP válidos (no nan, no vacíos)
        nuevos_sap_validos = nuevos_sap[
            (nuevos_sap.notna()) & 
            (nuevos_sap != 'nan') & 
            (nuevos_sap.str.strip() != '')
        ]
        
        # Actualizar solo los SAP válidos
        tiempos_limpio.loc[nuevos_sap_validos.index, 'SAP'] = nuevos_sap_validos
        
        filas_completadas = len(nuevos_sap_validos)
        
        # Identificar filas que fueron completadas
        filas_agregadas = tiempos_limpio[
            tiempos_limpio.index.isin(sap_vacios_antes.index) &
            ~(tiempos_limpio['SAP'].isna() |
              (tiempos_limpio['SAP'].astype(str).str.lower().isin(['nan'])) |
              (tiempos_limpio['SAP'].astype(str).str.strip() == '') |
              (tiempos_limpio['SAP'].astype(str).str.strip() == '0'))
        ].copy()
        
        print(f"\n✅ SAP completados: {filas_completadas} filas")
        
        if not filas_agregadas.empty:
            print(f"✅ FILAS CON SAP COMPLETADO: {len(filas_agregadas)} filas")
            print(f"\nEjemplo de filas completadas:")
            print(filas_agregadas[['Ruta', 'Nombre Ruta', 'SAP', 'Placa']].head(3))
        else:
            print("⚠️ No se pudieron completar SAP (verificar coincidencias de rutas)")
            
        print(f"\n--- DATAFRAME FINAL: tiempos_limpio ({len(tiempos_limpio)} filas) ---")
    else:
        print("❌ No se encontraron las columnas necesarias en resultados_completos")
        print(f"   Columna Ruta SARTT: {columna_ruta_sartt}")
        print(f"   Columna SAP SARTT: {columna_sap_sartt}")
        tiempos_limpio = df_filtrado_sf.copy()
else:
    print("❌ No se pudo ejecutar: Verificar que existan 'df_filtrado_sf' y 'resultados_completos'")
    if 'df_filtrado_sf' not in locals():
        print("   - Falta: df_filtrado_sf")
    if 'resultados_completos' not in locals():
        print("   - Falta: resultados_completos")
    elif resultados_completos.empty:
        print("   - resultados_completos está vacío")

Total de filas con SAP vacío antes: 1

Columnas disponibles en resultados_completos: ['Rutas_SART_SART', 'Rutas_CIAT_CIAT', 'Rutas_SAP_SAP', 'Rutas_Nombre_Nombre', 'Rutas_Tipo_Tipo', 'Rutas_Facturable_Peso kg', 'Rutas_Facturable_Volumen m3', 'Rutas_Entregable_Peso kg', 'Rutas_Entregable_Volumen m3', 'Rutas_Tiros_Tiros', 'Rutas_Partidas_Partidas', 'Rutas_Placa_Placa', 'Rutas_Línea_Línea', 'Rutas_Fecha Enrampe_Fecha Enrampe', 'Rutas_Anden_Anden', 'Rutas_Surtidores_Surtidores', 'Rutas_Autorizador_Autorizador', 'Rutas_Fecha Solicitud Transporte_Fecha Solicitud Transporte', 'Rutas_Planeación_Planeación', 'Rutas_En Patio_En Patio', 'Rutas_Asignación_Asignación', 'Rutas_Liberación_Liberación', 'Rutas_Fin Armado Paquete_Fin Armado Paquete', 'Rutas_En Ruta_En Ruta', 'Rutas_Fin de Ruta_Fin de Ruta', 'Rutas_Surtido_Inicio', 'Rutas_Surtido_Fin', 'Rutas_Carga_Inicio', 'Rutas_Carga_Fin', 'Rutas_Tiempo de retorno_Tiempo de retorno', 'Rutas_Segundo intento_Segundo intento', 'Rutas_Ruta Dinamica_Ruta D

8. Lectura Monitor Prueba (Inputs)

In [8]:
#Inputs para agregar sucursal y tipo ruta
import pandas as pd
import os
from datetime import datetime

# Diccionario de meses en español
meses_esp = {
    1: 'ene', 2: 'feb', 3: 'mar', 4: 'abr', 5: 'may', 6: 'jun',
    7: 'jul', 8: 'ago', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dic'
}

# Ruta del archivo Excel
archivo_excel = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Monitor Abastos\Concentrado Monitor.xlsx"

print("=== AGREGANDO COLUMNAS SUCURSAL, TIPO RUTA Y FECHAS DESDE HOJA INPUTS ===")

# Verificar que el archivo existe
if os.path.exists(archivo_excel):
    print(f"✅ Archivo encontrado: {os.path.basename(archivo_excel)}")
    
    # Leer la hoja "Inputs" - Columnas A, B, C (Nombre Ruta, Sucursal, TIPO)
    df_inputs = pd.read_excel(
        archivo_excel,
        sheet_name='Inputs',
        engine='openpyxl',
        usecols="A:C"
    )
    # Las columnas ya tienen los nombres correctos: 'Nombre Ruta', 'Sucursal', 'TIPO'
    print(f"Columnas leídas: {list(df_inputs.columns)}")
    # Eliminar duplicados por 'Nombre Ruta' para evitar problemas en el merge
    df_inputs = df_inputs.drop_duplicates(subset=['Nombre Ruta'])
    
    # Agregar la columna Sucursal y Tipo Ruta a tiempos_limpio usando merge tipo VLOOKUP
    if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
        tiempos_limpio = tiempos_limpio.merge(df_inputs[['Nombre Ruta', 'Sucursal']], on='Nombre Ruta', how='left')
        tiempos_limpio = tiempos_limpio.merge(df_inputs[['Nombre Ruta', 'TIPO']], on='Nombre Ruta', how='left')
        tiempos_limpio = tiempos_limpio.rename(columns={'TIPO': 'Tipo Ruta'})
        # Crear columnas de fecha usando dayfirst=True
        def formatear_fecha_cierre(valor):
            try:
                fecha = pd.to_datetime(valor, errors='coerce', dayfirst=True)
                if pd.isnull(fecha):
                    return ''
                return f"{fecha.day:02d}-{meses_esp[fecha.month]}"
            except:
                return ''
        def formatear_mes_cierre(valor):
            try:
                fecha = pd.to_datetime(valor, errors='coerce', dayfirst=True)
                if pd.isnull(fecha):
                    return ''
                return f"{meses_esp[fecha.month]}-{str(fecha.year)[-2:]}"
            except:
                return ''
        def obtener_ano(valor):
            try:
                fecha = pd.to_datetime(valor, errors='coerce', dayfirst=True)
                if pd.isnull(fecha):
                    return ''
                return f"{fecha.year}"
            except:
                return ''
        tiempos_limpio['Fecha Cierre'] = tiempos_limpio['Cierre'].apply(formatear_fecha_cierre)
        tiempos_limpio['Mes Cierre'] = tiempos_limpio['Cierre'].apply(formatear_mes_cierre)
        tiempos_limpio['Año'] = tiempos_limpio['Cierre'].apply(obtener_ano)
        print("✅ Columnas 'Sucursal', 'Tipo Ruta', 'Fecha Cierre', 'Mes Cierre' y 'Año' agregadas correctamente a tiempos_limpio.")
        print(f"Columnas finales: {list(tiempos_limpio.columns)}")
    else:
        print("❌ El DataFrame tiempos_limpio no existe o está vacío.")
else:
    print(f"❌ Archivo no encontrado: {archivo_excel}")


=== AGREGANDO COLUMNAS SUCURSAL, TIPO RUTA Y FECHAS DESDE HOJA INPUTS ===
✅ Archivo encontrado: Concentrado Monitor.xlsx
Columnas leídas: ['Nombre Ruta', 'Sucursal', 'TIPO']
✅ Columnas 'Sucursal', 'Tipo Ruta', 'Fecha Cierre', 'Mes Cierre' y 'Año' agregadas correctamente a tiempos_limpio.
Columnas finales: ['Tipo de Unidad', 'Ruta', 'Ruta Conca', 'Nombre Ruta', 'SAP', 'Placa', 'Cierre', 'Escaneos', 'Sucursal', 'Tipo Ruta', 'Fecha Cierre', 'Mes Cierre', 'Año']


9. Agregar columna Unidad desde Catalogo Consolidado

In [9]:
#Agregar columna unidad desde catalogo consolidado y SARTT
import pandas as pd
import os

archivo_catalogo = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Archivos de David Colin Domínguez - Catalogos de Placas\Consolidado catalogo de placas.xlsx"

print("=== AGREGANDO COLUMNA UNIDAD DESDE CATALOGO CONSOLIDADO Y SARTT ===")

if os.path.exists(archivo_catalogo):
    print(f"✅ Archivo encontrado: {os.path.basename(archivo_catalogo)}")
    # Leer la hoja 'Catalogo consolidado'
    df_catalogo = pd.read_excel(
        archivo_catalogo,
        sheet_name='Catalogo consolidado',
        engine='openpyxl'
    )
    # Asegurar que las columnas relevantes existan
    if 'Placa' in df_catalogo.columns and 'Unidad' in df_catalogo.columns:
        # Eliminar duplicados por 'Placa' para evitar problemas en el merge
        df_catalogo = df_catalogo.drop_duplicates(subset=['Placa'])
        # Agregar la columna Unidad a tiempos_limpio usando merge tipo VLOOKUP
        if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
            tiempos_limpio = tiempos_limpio.merge(df_catalogo[['Placa', 'Unidad']], on='Placa', how='left')
            # Si no encuentra coincidencia, poner 'Sin unidad'
            tiempos_limpio['Unidad'] = tiempos_limpio['Unidad'].fillna('Sin unidad')
            print("✅ Columna 'Unidad' agregada correctamente a tiempos_limpio.")

            # --- MODIFICACIÓN: Cambiar valores específicos en Unidad ---
            tiempos_limpio['Unidad'] = tiempos_limpio['Unidad'].replace({
                '53 ft': 'C53',
                '40 ft H.C': 'C40',
                '48 ft': 'C48'
            })

            # --- NUEVO FLUJO: Buscar en SARTT solo las rutas con 'Sin unidad' ---
            sin_unidad = tiempos_limpio[tiempos_limpio['Unidad'] == 'Sin unidad'].copy()
            rutas_sin_unidad = sin_unidad['Ruta'].dropna().unique().tolist()
            if rutas_sin_unidad:
                print(f"Rutas con 'Sin unidad': {rutas_sin_unidad}")
                rutas_texto = '\n'.join(map(str, rutas_sin_unidad))

                # --- Código Selenium para buscar en SARTT (robusto con fallback a Chrome) ---
                import time
                from selenium import webdriver
                from selenium.webdriver.edge.service import Service as EdgeService
                from selenium.webdriver.chrome.service import Service as ChromeService
                from selenium.webdriver.common.by import By
                from selenium.webdriver.common.keys import Keys
                from selenium.webdriver.support.ui import WebDriverWait, Select
                from selenium.webdriver.support import expected_conditions as EC
                from selenium.common.exceptions import TimeoutException, NoSuchDriverException, WebDriverException

                USUARIO = "IGCAMPOSG"
                CONTRASENA = "Carcachita22$"
                URLS_LOGIN = [
                    "https://sartt.truper.com/sartt/login.jspx",
                    "http://sartt.truper.com/sartt/login.jspx"
                ]
                URLS_REPORTE = [
                    "https://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx",
                    "http://sartt.truper.com/sartt/reportes/reporteRecuperacionUnidadesPrincipal.jspx"
                ]

                def crear_driver_unidad():
                    """Intenta Edge con rutas locales del driver y usa Chrome como respaldo."""
                    edge_options = webdriver.EdgeOptions()
                    edge_options.add_argument("--start-maximized")
                    edge_options.add_argument("--ignore-certificate-errors")
                    edge_options.add_argument("--inprivate")

                    rutas_edge_driver = [
                        os.environ.get('MSEDGEDRIVER_PATH'),
                        r"C:\WebDriver\msedgedriver.exe",
                        r"C:\tools\msedgedriver.exe",
                        r"C:\Program Files\msedgedriver.exe",
                        r"C:\Program Files (x86)\msedgedriver.exe"
                    ]
                    for ruta in rutas_edge_driver:
                        if ruta and os.path.exists(ruta):
                            try:
                                print(f"Usando EdgeDriver local: {ruta}")
                                return webdriver.Edge(service=EdgeService(executable_path=ruta), options=edge_options), 'edge'
                            except Exception as exc:
                                print(f"No se pudo iniciar Edge con {ruta}: {exc}")

                    try:
                        return webdriver.Edge(options=edge_options), 'edge'
                    except (NoSuchDriverException, WebDriverException) as exc:
                        print(f"No se pudo iniciar Edge automatico: {exc}")
                        print("Fallback a Chrome para no detener el proceso...")
                        chrome_options = webdriver.ChromeOptions()
                        chrome_options.add_argument("--start-maximized")
                        chrome_options.add_argument("--ignore-certificate-errors")
                        return webdriver.Chrome(service=ChromeService(), options=chrome_options), 'chrome'

                driver, navegador_unidad = crear_driver_unidad()
                print(f"Navegador Selenium activo: {navegador_unidad}")
                wait = WebDriverWait(driver, 30)
                resultados_completos_unidad = pd.DataFrame()

                def abrir_url_robusta_local(urls, etiqueta):
                    ultimo_error = None
                    for intento, url in enumerate(urls, start=1):
                        try:
                            driver.get(url)
                            wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
                            time.sleep(1)
                            print(f"{etiqueta} intento {intento}: {url}")
                            print(f"URL actual: {driver.current_url} | Titulo: {driver.title}")
                            if not driver.current_url.startswith("data:"):
                                return
                        except Exception as exc:
                            ultimo_error = exc
                    raise TimeoutException(f"No fue posible abrir {etiqueta} sin quedar en data:. Ultimo error: {ultimo_error}")

                try:
                    abrir_url_robusta_local(URLS_LOGIN, "login SART unidad")
                    wait.until(EC.element_to_be_clickable((By.ID, "loginForm:j_username"))).send_keys(USUARIO)
                    wait.until(EC.element_to_be_clickable((By.ID, "loginForm:j_password"))).send_keys(CONTRASENA)
                    boton_login_unidad = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[id^='loginForm:j_id'], input.iceCmdBtn[value='Entrar'], input[type='submit'][value='Entrar']")))
                    try:
                        boton_login_unidad.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", boton_login_unidad)
                    wait.until(lambda d: (not d.current_url.startswith("data:")) and (("dashboard" in d.page_source.lower()) or ("reporte" in d.current_url.lower()) or ("principal" in d.current_url.lower())))

                    abrir_url_robusta_local(URLS_REPORTE, "reporte SART unidad")

                    try:
                        select_element = wait.until(EC.element_to_be_clickable((By.ID, "forma:icePnlTbSet:0:j_id202")))
                    except Exception:
                        select_element = wait.until(EC.element_to_be_clickable((By.ID, "forma:icePnlTbSet:0:j_id200")))
                    Select(select_element).select_by_value("2")  # Selecciona CIAT
                    time.sleep(1)
                    try:
                        cuadro_rutas = wait.until(EC.element_to_be_clickable((By.ID, "forma:icePnlTbSet:0:j_id206")))
                    except Exception:
                        cuadro_rutas = wait.until(EC.element_to_be_clickable((By.ID, "forma:icePnlTbSet:0:j_id204")))
                    cuadro_rutas.clear()
                    cuadro_rutas.send_keys(rutas_texto)
                    time.sleep(1)
                    cuadro_rutas.send_keys(Keys.TAB)
                    time.sleep(5)  # Espera para que el usuario pueda dar clic en Buscar

                    # El usuario debe dar clic manualmente en Buscar
                    id_tabla_resultados = "forma:icePnlTbSet:0:rutas"
                    wait.until(EC.visibility_of_element_located((By.ID, id_tabla_resultados)))
                    tabla_elemento = driver.find_element(By.ID, id_tabla_resultados)
                    tabla_html = tabla_elemento.get_attribute('outerHTML')
                    from io import StringIO
                    resultados_completos_unidad = pd.read_html(StringIO(tabla_html))[0]
                    if isinstance(resultados_completos_unidad.columns, pd.MultiIndex):
                        resultados_completos_unidad.columns = ['_'.join(col).strip() for col in resultados_completos_unidad.columns.values]
                    resultados_completos_unidad.columns = resultados_completos_unidad.columns.str.replace(r'[^\w\s]', '', regex=True)
                    resultados_completos_unidad.columns = resultados_completos_unidad.columns.str.strip()
                    print("--- RESULTADOS SARTT PARA UNIDAD ---")
                except TimeoutException as e:
                    print("❌ Error: Se agotó el tiempo de espera.")
                except Exception as e:
                    print(f"❌ Error inesperado: {e}")
                finally:
                    if 'driver' in locals():
                        driver.quit()

                # --- Completar Placa y Unidad usando los resultados de SARTT ---
                if not sin_unidad.empty and not resultados_completos_unidad.empty:
                    columna_ruta_sartt = None
                    columna_placa_sartt = None
                    columna_unidad_sartt = None
                    for col in resultados_completos_unidad.columns:
                        if 'CIAT' in col.upper() and 'RUTA' in col.upper():
                            columna_ruta_sartt = col
                        elif 'PLACA' in col.upper():
                            columna_placa_sartt = col
                        elif 'UNIDAD' in col.upper():
                            columna_unidad_sartt = col
                    if columna_ruta_sartt and columna_placa_sartt and columna_unidad_sartt:
                        mapeo_placa = dict(zip(resultados_completos_unidad[columna_ruta_sartt].astype(str), resultados_completos_unidad[columna_placa_sartt].astype(str)))
                        mapeo_unidad = dict(zip(resultados_completos_unidad[columna_ruta_sartt].astype(str), resultados_completos_unidad[columna_unidad_sartt].astype(str)))
                        filas_actualizadas = 0
                        for idx, row in tiempos_limpio.iterrows():
                            if row['Unidad'] == 'Sin unidad':
                                ruta_actual = str(row['Ruta'])
                                if ruta_actual in mapeo_placa:
                                    tiempos_limpio.at[idx, 'Placa'] = mapeo_placa[ruta_actual]
                                if ruta_actual in mapeo_unidad:
                                    nueva_unidad = mapeo_unidad[ruta_actual]
                                    if nueva_unidad and nueva_unidad.strip() != '':
                                        tiempos_limpio.at[idx, 'Unidad'] = nueva_unidad
                                        filas_actualizadas += 1
                        print(f"✅ Filas actualizadas con datos de SARTT: {filas_actualizadas}")
                        # --- MODIFICACIÓN: Cambiar valores específicos en Unidad después de completar ---
                        tiempos_limpio['Unidad'] = tiempos_limpio['Unidad'].replace({
                            '53 ft': 'C53',
                            '40 ft H.C': 'C40',
                            '48 ft': 'C48'
                        })
                    else:
                        print("❌ No se encontraron las columnas necesarias en resultados_completos_unidad para Placa y Unidad.")

            print(f"Columnas finales: {list(tiempos_limpio.columns)}")
        else:
            print("❌ El DataFrame tiempos_limpio no existe o está vacío.")
    else:
        print("❌ No se encontraron las columnas 'Placa' y 'Unidad' en el catálogo.")
else:
    print(f"❌ Archivo no encontrado: {archivo_catalogo}")

# Asegurar que 'SAP' sea string antes de cualquier merge usando 'SAP'
if 'SAP' in tiempos_limpio.columns:
    tiempos_limpio['SAP'] = tiempos_limpio['SAP'].astype(str)
if 'df_catalogo' in locals() and 'SAP' in df_catalogo.columns:
    df_catalogo['SAP'] = df_catalogo['SAP'].astype(str)

# Eliminar filas duplicadas según la columna 'Ruta Conca', conservando solo la primera aparición
tiempos_limpio = tiempos_limpio.drop_duplicates(subset=['Ruta Conca'])


=== AGREGANDO COLUMNA UNIDAD DESDE CATALOGO CONSOLIDADO Y SARTT ===
✅ Archivo encontrado: Consolidado catalogo de placas.xlsx
✅ Columna 'Unidad' agregada correctamente a tiempos_limpio.
Rutas con 'Sin unidad': ['73429', '72798', '73476', '73607', '72143', '73101', '72430', '72426', '73483', '73389', '73601', '72764', '73384', '73107', '72148', '72427', '73105', '72153', '72782', '72767', '73034', '73033', '73086', '72052', '73038', '73772', '72808', '72840', '72454', '72762', '73149', '73085', '73424', '72792', '72763', '73381', '73738', '73753', '72129', '72834', '73110', '72453']
No se pudo iniciar Edge automatico: Message: Unable to obtain driver for MicrosoftEdge; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location

Fallback a Chrome para no detener el proceso...
Navegador Selenium activo: chrome
login SART unidad intento 2: http://sartt.truper.com/sartt/login.jspx
URL actual: http://sartt.truper.com

10. ZSR400

In [10]:
#Esto si estoy pensando en borarrlo
import pandas as pd
import os
from datetime import datetime

# Carpeta donde están los archivos
carpeta = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\ZSR400"

# Carpeta de salida
output_folder = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs"

# Fecha y hora actual para el nombre del archivo
fecha_hora = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_folder, f"ZSR400_limpio_{fecha_hora}.csv")

# Encabezados personalizados
columnas = [
    "Transporte", "Denomin.", "Sucursal", "Costo", "Volumen/m3",
    "Peso neto/", "Fecha", "Flete", "% Flete", "Transportista"
]

ZSR400 = pd.DataFrame()

if not os.path.exists(carpeta):
    print(f"⚠️ Carpeta ZSR400 no encontrada, se omite este paso: {carpeta}")
else:
    # Lista para guardar los DataFrames
    dfs = []

    # Itera sobre todos los archivos en la carpeta
    for archivo in os.listdir(carpeta):
        ruta = os.path.join(carpeta, archivo)
        if os.path.isfile(ruta) and archivo.lower().endswith('.xls'):
            df = pd.read_csv(
                ruta,
                sep='\t',
                skiprows=9,
                names=columnas,
                header=None,
                encoding='latin1'
            )
            df = df[columnas]  # Elimina columnas sin encabezado
            df = df.dropna(axis=1, how='all')  # Elimina columnas vacías
            df = df.dropna(subset=['Transporte'])  # Elimina filas donde 'Transporte' esté vacía
            df = df[df['Transporte'] != 'Transporte']  # Elimina encabezados repetidos
            df['Archivo'] = archivo  # Agrega columna con el nombre del archivo
            dfs.append(df)

    if not dfs:
        print("⚠️ No se encontraron archivos .xls en la carpeta ZSR400, se omite este paso.")
    else:
        # Une todos los DataFrames
        ZSR400 = pd.concat(dfs, ignore_index=True)

        # Eliminar filas duplicadas por la columna 'Transporte'
        ZSR400 = ZSR400.drop_duplicates(subset=['Transporte'])

        # Guarda el resultado en CSV con fecha y hora
        ZSR400.to_csv(output_path, index=False, encoding='latin1')
        print(f"✅ Archivo ZSR400 generado y guardado en: {output_path} (sin duplicados en Transporte)")


✅ Archivo ZSR400 generado y guardado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs\ZSR400_limpio_20260511_100845.csv (sin duplicados en Transporte)


11. Imprimir CSV

In [11]:
#Imprimir y guardar tiempos_limpio
import os
from datetime import datetime

# Verificar que tenemos el DataFrame tiempos_limpio
if 'tiempos_limpio' in locals() and not tiempos_limpio.empty:
    
    # Si existe la columna 'Unidad', renombrarla a 'Unidad catalogo' para el CSV final
    if 'Unidad' in tiempos_limpio.columns:
        tiempos_limpio = tiempos_limpio.rename(columns={'Unidad': 'Unidad catalogo'})

    # Ruta de la carpeta de salida
    ruta_outputs = r"C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs"
    
    # Crear la carpeta si no existe
    os.makedirs(ruta_outputs, exist_ok=True)
    
    # Generar nombre base con fecha y hora
    fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_base = f"TiemposRuta_Limpio_{fecha_actual}"
    extension = ".csv"
    
    # Función para encontrar el siguiente número consecutivo
    def obtener_nombre_unico(ruta_carpeta, nombre_base, extension):
        nombre_archivo = f"{nombre_base}{extension}"
        ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
        
        # Si el archivo no existe, usar el nombre original
        if not os.path.exists(ruta_completa):
            return nombre_archivo, ruta_completa
        
        # Si existe, buscar el siguiente número consecutivo
        contador = 1
        while True:
            nombre_archivo = f"{nombre_base}({contador}){extension}"
            ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
            
            if not os.path.exists(ruta_completa):
                return nombre_archivo, ruta_completa
            
            contador += 1
    
    # Obtener el nombre único
    nombre_archivo, ruta_completa = obtener_nombre_unico(ruta_outputs, nombre_base, extension)

    # Asegurar que ambas columnas existan y no se sobrescriban
    columnas_orden = [
        'Tipo de Unidad',
        'Ruta',
        'Ruta Conca',
        'Nombre Ruta',
        'SAP',
        'Placa',
        'Cierre',
        'Escaneos',
        'Sucursal',
        'Unidad catalogo',      # Unidad del catálogo
        'Tipo Ruta',
        'Fecha Cierre',
        'Mes Cierre',
        'Año',
        'Volumen Cierre SAP',
        'Peso Cierre SAP',
        'Volumen plan',
        'Peso plan',
        'Unidad planeada',      # Unidad del merge LP
        'Tipo',
    ]
    # Solo incluir columnas que existan en el DataFrame
    columnas_finales = [col for col in columnas_orden if col in tiempos_limpio.columns]
    tiempos_limpio = tiempos_limpio.reindex(columns=columnas_finales)

    # Guardar el DataFrame en CSV
    try:
        tiempos_limpio.to_csv(ruta_completa, index=False, encoding='utf-8-sig')
        print(f"✅ Archivo guardado exitosamente:")
        print(f"   📁 Carpeta: {ruta_outputs}")
        print(f"   📄 Archivo: {nombre_archivo}")
        print(f"   📊 Filas: {len(tiempos_limpio)}")
        print(f"   🕒 Fecha de guardado: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        
    except Exception as e:
        print(f"❌ Error al guardar el archivo: {e}")
        
else:
    print("❌ No se encontró el DataFrame tiempos_limpio o está vacío")
    if 'tiempos_limpio' not in locals():
        print("   El DataFrame tiempos_limpio no ha sido creado")
    elif tiempos_limpio.empty:
        print("   El DataFrame tiempos_limpio está vacío")

✅ Archivo guardado exitosamente:
   📁 Carpeta: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Limpieza TiemposRuta Abastos\Outputs
   📄 Archivo: TiemposRuta_Limpio_20260511_100845.csv
   📊 Filas: 288
   🕒 Fecha de guardado: 2026-05-11 10:08:45
